# Package 3 — Predict upsell (classification)

Target: `upsell` (binary: 0/1). Compare XGBoost, LightGBM, CatBoost with 5-fold CV.

## Feature selection — redone for this target

Leakage depends on the target, so this has to be reconsidered from scratch, not copied
from Package 2.

Excluded as leakage: `cumulative_profit`, `ltv_months`, `referred` — all three unfold
*during* the same customer relationship period as `upsell` itself, so they are not known
at the moment we'd want to predict whether a customer will upsell.

Kept: `purchased` (marks the start of the relationship) plus all funnel/acquisition
columns (`ad_budget`, `num_leads`, `leads_answered`, `leads_not_answered`,
`followup_1..5`, `not_closed`, `closed`, `calls_to_closed`, `calls_to_not_closed`,
`customer_acquisition_cost`) — same reasoning as Package 2: these happen before/at the
start of the relationship.

No missing values in `upsell` (0/1, 2034/1466 — reasonably balanced), so no rows to drop.

In [1]:
import pandas as pd

df = pd.read_csv("../data/funnel_marketing_data.csv")

features = [
    "ad_budget", "num_leads", "leads_answered", "leads_not_answered",
    "followup_1", "followup_2", "followup_3", "followup_4", "followup_5",
    "not_closed", "closed", "calls_to_closed", "calls_to_not_closed",
    "customer_acquisition_cost", "purchased",
]

X = df[features]
y = df["upsell"]
X.shape, y.shape, y.value_counts()

((3500, 15),
 (3500,),
 upsell
 0    2034
 1    1466
 Name: count, dtype: int64)

## Cross-validation setup

`StratifiedKFold` instead of plain `KFold`: keeps the class ratio (58% / 42%) roughly
the same in every fold, so no fold is accidentally skewed toward one class.

For each model we track both threshold-based metrics (accuracy, precision, recall, F1
— computed on the 0/1 prediction) and `roc_auc` (computed on the predicted probability,
no threshold involved — measures ranking quality).

In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=False),
}

metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc"]
results = {name: {m: [] for m in metric_names} for name in models}

for train_idx, test_idx in skf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        results[name]["accuracy"].append(accuracy_score(y_test, preds))
        results[name]["precision"].append(precision_score(y_test, preds))
        results[name]["recall"].append(recall_score(y_test, preds))
        results[name]["f1"].append(f1_score(y_test, preds))
        results[name]["roc_auc"].append(roc_auc_score(y_test, probs))

summary = pd.DataFrame({
    name: {m: np.mean(scores[m]) for m in metric_names}
    for name, scores in results.items()
}).T

summary.round(3)

,accuracy,precision,recall,f1,roc_auc
XGBoost,0.749,0.680,0.757,0.716,0.800
LightGBM,0.762,0.685,0.799,0.738,0.808
CatBoost,0.784,0.704,0.835,0.764,0.821


## A concrete confusion matrix

One 80/20 stratified split with CatBoost, to see actual counts behind the metrics
above (not just cross-validated averages).

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cm_model = CatBoostClassifier(random_state=42, verbose=False)
cm_model.fit(X_train, y_train)
cm_preds = cm_model.predict(X_test)

cm = confusion_matrix(y_test, cm_preds)
pd.DataFrame(
    cm,
    index=["Actual: no upsell (0)", "Actual: upsell (1)"],
    columns=["Predicted: no (0)", "Predicted: yes (1)"],
)

,Predicted: no (0),Predicted: yes (1)
Actual: no upsell (0),301,106
Actual: upsell (1),51,242


## Feature importance comparison

Same approach as Package 2: refit each model on the full dataset for a stable
importance estimate, normalize each column to sum to 100% since the three libraries
compute importance differently and are not on the same scale.

In [4]:
for model in models.values():
    model.fit(X, y)

importances = pd.DataFrame({
    name: model.feature_importances_
    for name, model in models.items()
}, index=X.columns)

importances_pct = importances.div(importances.sum(axis=0), axis=1) * 100
importances_pct.round(1).sort_values("CatBoost", ascending=False)

,XGBoost,LightGBM,CatBoost
purchased,63.1,1.5,36.3
calls_to_closed,13.6,6.6,16.3
customer_acquisition_cost,4.5,8.2,8.5
leads_not_answered,1.5,11.5,4.6
calls_to_not_closed,1.4,6.7,4.5
leads_answered,1.8,11.8,4.3
num_leads,1.6,12.5,3.8
followup_1,1.7,9.2,3.6
followup_2,1.4,7.3,3.2
followup_3,1.5,7.0,2.9


## Refined problem: predict upsell among purchasers only

`purchased=0` guarantees `upsell=0` (337 rows, no exceptions) — correct business logic,
not leakage, but it means part of the model's apparent performance above comes from
solving the easy sub-problem "did they even buy." The real business question is: among
customers who already purchased, who will take the upsell? Filtering to `purchased=1`
and dropping `purchased` from the features (constant now, carries no information) gives
the harder, more realistic version of the problem.

In [5]:
df_purchasers = df[df["purchased"] == 1]

features_v2 = [f for f in features if f != "purchased"]
X2 = df_purchasers[features_v2]
y2 = df_purchasers["upsell"]

X2.shape, y2.shape, y2.value_counts()

((3163, 14),
 (3163,),
 upsell
 0    1697
 1    1466
 Name: count, dtype: int64)

Rerunning the same 5-fold stratified CV, same 3 models, same metrics — on this
harder, purchasers-only version of the problem.

In [6]:
models_v2 = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=False),
}

results_v2 = {name: {m: [] for m in metric_names} for name in models_v2}

for train_idx, test_idx in skf.split(X2, y2):
    X_train, X_test = X2.iloc[train_idx], X2.iloc[test_idx]
    y_train, y_test = y2.iloc[train_idx], y2.iloc[test_idx]

    for name, model in models_v2.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]

        results_v2[name]["accuracy"].append(accuracy_score(y_test, preds))
        results_v2[name]["precision"].append(precision_score(y_test, preds))
        results_v2[name]["recall"].append(recall_score(y_test, preds))
        results_v2[name]["f1"].append(f1_score(y_test, preds))
        results_v2[name]["roc_auc"].append(roc_auc_score(y_test, probs))

summary_v2 = pd.DataFrame({
    name: {m: np.mean(scores[m]) for m in metric_names}
    for name, scores in results_v2.items()
}).T

summary_v2.round(3)

,accuracy,precision,recall,f1,roc_auc
XGBoost,0.717,0.675,0.749,0.710,0.756
LightGBM,0.732,0.682,0.789,0.731,0.766
CatBoost,0.755,0.698,0.832,0.759,0.781


## Feature importance — purchasers-only model

Refit on the full purchasers-only dataset for a stable estimate.

In [7]:
for model in models_v2.values():
    model.fit(X2, y2)

importances_v2 = pd.DataFrame({
    name: model.feature_importances_
    for name, model in models_v2.items()
}, index=X2.columns)

importances_v2_pct = importances_v2.div(importances_v2.sum(axis=0), axis=1) * 100
importances_v2_pct.round(1).sort_values("CatBoost", ascending=False)

,XGBoost,LightGBM,CatBoost
calls_to_closed,43.5,6.8,26.1
customer_acquisition_cost,10.9,8.8,13.5
leads_not_answered,3.5,11.9,7.3
calls_to_not_closed,3.4,6.5,7.3
leads_answered,3.7,11.0,7.1
followup_1,4.1,9.5,5.8
num_leads,4.1,12.6,5.7
followup_3,3.7,7.0,5.0
followup_2,3.9,7.4,4.7
followup_5,3.9,4.7,4.3


## Conclusions

**Chosen model: CatBoost.** Best across every metric in both versions of the problem.
On the refined, purchasers-only problem: accuracy 0.755, precision 0.698, recall 0.832, F1 0.759, ROC-AUC 0.781.

**Key finding — a structural fact, not leakage:** `purchased=0` guarantees `upsell=0` (337/337 rows, no exceptions) — correct business logic (you can't upsell someone who never bought), but it meant the first version of this model was partly solving the easy sub-problem of "did they buy at all." Refiltering to purchasers only (`purchased=1`, `purchased` dropped from features) is the real business question, and is harder: ROC-AUC drops from 0.821 to 0.781, accuracy from 0.784 to 0.755. Both numbers are still well above the naive baseline (53.6% accuracy from always guessing the majority class on this subset).

**Cross-package finding:** `calls_to_closed` (fewer calls needed to close the initial sale) is the top driver of *both* longer LTV (Package 2, correlation -0.65) and upsell likelihood (Package 3, 26-44% of importance). Same signal showing up twice, independently, in two different target variables — a real, repeatable pattern rather than noise in one model.

**Business takeaway for Northbound Media:** deployed predictions should be *probabilities*, not a hard yes/no — rank purchasers by predicted upsell probability and let the sales team work down the list, rather than only calling everyone the model labels "1" at the default 0.5 threshold. Given a missed upsell opportunity (false negative) likely costs more than one unproductive call (false positive), leaning the decision threshold toward recall is worth considering once this ships.